# Distributed Inference

## Learning Objectives
1. Understand tensor, pipeline, and sequence parallelism
2. Compute bubble overhead and efficiency
3. Analyze communication vs compute trade-offs
4. Choose parallelism strategy for your model size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('Distributed inference simulation (TP, PP, SP)')

## Level 1: Tensor Parallelism Basics

In [ ]:
# Level 1: Simulate tensor parallelism (within node)
# TP splits attention heads across GPUs

def simulate_tensor_parallel_inference(model_dim=4096, num_heads=32, seq_len=2048,
                                       tp_degree=4, compute_time_per_head=10):
    '''Simulate single token generation with tensor parallelism.'''
    
    heads_per_gpu = num_heads // tp_degree
    
    # Each GPU computes its heads in parallel
    per_gpu_compute = heads_per_gpu * compute_time_per_head
    
    # AllReduce to aggregate: O(model_dim) with AllReduce bandwidth
    allreduce_bandwidth = 600  # GB/s (NVLink)
    allreduce_size = model_dim * 4 / 1e9  # fp32
    allreduce_time = allreduce_size / allreduce_bandwidth * 1000  # ms
    
    total_time = per_gpu_compute + allreduce_time
    utilization = per_gpu_compute / total_time
    
    return total_time, utilization, allreduce_time

# Test different TP degrees
tp_degrees = [1, 2, 4, 8]
times_tp = []
utils_tp = []

print('Tensor Parallelism (within-node with NVLink):')
print('-' * 60)
print('TP Degree | Compute Time | AllReduce | Total | Util')
print('-' * 60)

for tp_deg in tp_degrees:
    time_total, util, time_ar = simulate_tensor_parallel_inference(tp_degree=tp_deg)
    times_tp.append(time_total)
    utils_tp.append(util)
    print(f'  {tp_deg:1d}      | {10*32/tp_deg:6.1f} ms   | {time_ar:6.2f} ms | {time_total:6.2f} ms | {util:5.1%}')

print('\nKey insight: TP saturates AllReduce bandwidth around TP=4-8')

## Level 2: Pipeline Parallelism with Bubble Overhead

In [ ]:
# Level 2: Pipeline parallelism bubble overhead
# PP splits layers across GPUs: GPU0=layers 0-L/P, GPU1=layers L/P-2L/P, etc

def compute_pipeline_bubble(num_pipeline_stages, micro_batch_count):
    '''Compute pipeline bubble fraction.
    
    Formula: bubble = (P - 1) / (M + P - 1)
    P: pipeline depth (number of GPUs)
    M: micro-batch count (batches per pipeline stage)
    '''
    P = num_pipeline_stages
    M = micro_batch_count
    
    bubble = (P - 1) / (M + P - 1)
    utilization = 1 - bubble
    
    return bubble, utilization

# Matrix of bubble overhead
pipeline_stages = [2, 4, 8]
micro_batches = [1, 2, 4, 8, 16]

print('\nPipeline Parallelism Bubble Overhead:')
print('(Lower bubbles are better)')
print('-' * 70)
print('Stages \ M-batches', end='')
for m in micro_batches:
    print(f'  M={m:2d}', end='')
print()
print('-' * 70)

for P in pipeline_stages:
    print(f'  P={P}       ', end='')
    for M in micro_batches:
        bubble, util = compute_pipeline_bubble(P, M)
        print(f'  {util:5.0%} ', end='')
    print()

print('-' * 70)
print('Recommendation: Use M >= P*2 to keep bubble < 25%')

# Plot bubble vs micro-batch count
fig, ax = plt.subplots(figsize=(10, 5))

m_range = np.arange(1, 32)
for p in [2, 4, 8]:
    bubbles = [(p - 1) / (m + p - 1) for m in m_range]
    ax.plot(m_range, [1-b for b in bubbles], marker='o', linewidth=2.5, label=f'P={p}', markersize=6)

ax.axhline(0.75, color='green', linestyle='--', linewidth=2, alpha=0.5, label='75% util threshold')
ax.set_xlabel('Micro-Batch Count (M)')
ax.set_ylabel('Pipeline Utilization')
ax.set_title('Pipeline Parallelism: Achieving High Utilization')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim([0.5, 1.0])
plt.tight_layout()
plt.show()

## Real-World Example 1: 70B Model on 8 A100s

In [ ]:
# Example 1: Distribute LLaMA-70B on 8xA100 (80GB each)
# Strategy: Use both TP and PP

model_params_b = 70  # 70B parameters
param_bytes_fp16 = 2  # bytes per parameter in fp16
total_model_memory_gb = model_params_b * param_bytes_fp16  # 140GB

# Strategy 1: TP only (tp_degree=4, two nodes with NVLink)
print('70B LLaMA on 8xA100 (80GB each):')
print('='*60)
print('\nStrategy 1: Pure Tensor Parallelism (TP=4)')
print('-'*60)
tp = 4
model_per_gpu = total_model_memory_gb / tp
batch_size_tp4 = 8
per_request_kv_cache = 50  # MB
max_requests = (80 - model_per_gpu) / per_request_kv_cache

print(f'Model per GPU: {model_per_gpu:.1f} GB')
print(f'Available for KV cache: {80 - model_per_gpu:.1f} GB')
print(f'Max concurrent requests: ~{int(max_requests)} (50MB KV cache each)')
print(f'Throughput: {batch_size_tp4 * 4:.0f} tokens/sec')
print(f'Latency (batch=8, TP=4): ~150ms')

# Strategy 2: TP=4 + PP=2 (split between two nodes)
print('\nStrategy 2: Mixed TP=4 + PP=2 (intra-node TP, inter-node PP)')
print('-'*60)
pp = 2
tp2 = 4
model_per_gpu = total_model_memory_gb / (tp2 * pp)
max_requests_pp = (80 - model_per_gpu) / per_request_kv_cache

# Compute bubble
M = 8  # micro batches
bubble, util = compute_pipeline_bubble(pp, M)

print(f'TP=4 (within node), PP=2 (across nodes)')
print(f'Model per GPU: {model_per_gpu:.1f} GB')
print(f'Max concurrent requests: ~{int(max_requests_pp)} (more KV cache capacity)')
print(f'Bubble overhead: {bubble:.0%} (utilization: {util:.0%})')
print(f'Throughput loss due to bubble: ~{bubble*100:.0f}%')
print(f'Effective throughput: {int(batch_size_tp4 * 4 * util):.0f} tokens/sec')

# Strategy 3: TP=2 + PP=4
print('\nStrategy 3: TP=2 + PP=4 (more inter-node, less bubble)')
print('-'*60)
pp3 = 4
tp3 = 2
bubble3, util3 = compute_pipeline_bubble(pp3, M)

print(f'TP=2 (pairs of nodes), PP=4 (across all nodes)')
print(f'Bubble overhead: {bubble3:.0%} (utilization: {util3:.0%})')
print(f'Effective throughput: {int(batch_size_tp4 * 4 * util3):.0f} tokens/sec')
print(f'NOTE: Higher latency due to slower inter-node communication')

print('\nRecommendation: Use TP=4 + PP=2 for 8xA100')
print('  Pro: Fast intra-node communication, parallelism across nodes')
print('  Con: Slower inter-node links, so keep PP small')

## Real-World Example 2: TP vs PP Trade-off

In [ ]:
# Example 2: Analyze TP vs PP trade-offs for different model sizes

model_sizes_b = [7, 13, 34, 70]
num_gpus = [4, 4, 8, 8]

print('Parallelism Strategy Recommendation by Model Size:')
print('='*70)
print('Model | Num GPUs | Strategy | Rationale')
print('='*70)

strategies = {
    7: 'TP=4 (no PP needed)',
    13: 'TP=4 (fits on single node)',
    34: 'TP=4 + PP=2 (split nodes, TP for speed)',
    70: 'TP=4 + PP=2 or TP=2 + PP=4 (balance)',
}

for size, num_gpu in zip(model_sizes_b, num_gpus):
    model_memory_gb = size * 2
    per_gpu_gb = model_memory_gb / num_gpu
    
    print(f'{size:3d}B | {num_gpu:7d}  | {strategies[size]:20s} | ', end='')
    
    if size <= 13:
        print(f'{per_gpu_gb:.0f}GB per GPU (TP saturates at 4)')
    else:
        bubble, util = compute_pipeline_bubble(2, 8)
        print(f'{per_gpu_gb:.0f}GB per GPU, bubble={bubble:.0%}')

print('='*70)

## Key Takeaways

In [ ]:
print('='*70)
print('DISTRIBUTED INFERENCE BEST PRACTICES')
print('='*70)
print('')
print('1. TENSOR PARALLELISM (TP)')
print('   - Within a node, via NVLink (600 GB/s)')
print('   - Ideal degree: 4-8 (balance compute vs AllReduce)')
print('   - Overhead: ~10-20% for TP=4')
print('')
print('2. PIPELINE PARALLELISM (PP)')
print('   - Across nodes, via slower interconnect')
print('   - Use micro-batching: M >= 2*P to reduce bubble')
print('   - Overhead: Keep bubble < 25% (use M >= 16 for P=4)')
print('')
print('3. SEQUENCE PARALLELISM (SP)')
print('   - Rarely used for inference (prefill-bounded)')
print('   - More relevant for training with long sequences')
print('')
print('4. COMBINED TP+PP')
print('   - Use TP=4 (within node) + PP=2-4 (across nodes)')
print('   - Example: 70B on 8xA100 → TP=4 + PP=2')
print('   - Communication cost: TP >> PP (keep TP fast)')
print('')
print('5. AVOIDING PITFALLS')
print('   - Don\'t use TP alone for models >13B (need PP)')
print('   - Don\'t skip micro-batching in PP (bubble blows up)')
print('   - Don\'t overfit TP (AllReduce becomes bottleneck)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Bubble Optimization')
print('-' * 70)
print('For a 4-stage pipeline on 100 inference requests:')
print('  - Compute bubble for M=1, 2, 4, 8, 16')
print('  - Which M minimizes total time?')
print('  - What\'s the memory overhead of using large M?')
print('')
print('Exercise 2: Communication Modeling')
print('-' * 70)
print('Model: TP AllReduce = O(model_dim) * (link_bandwidth)')
print('If you upgrade from PCIe to NVLink:')
print('  - Current: 80 GB/s PCIe')
print('  - New: 600 GB/s NVLink')
print('  - How much faster is TP=4 inference?')
print('')
print('Success! Generated notebook 51 (distributed-inference)')